# Модуль 1. Сохраняем и загружаем модели: `dump` и `load`

## Зачем этот модуль

В предыдущем модуле мы научились сохранять простые объекты — словари и массивы. Но в машинном обучении главный объект — это **обученная модель**. Обучение может занимать часы, дни или недели. Если после обучения просто закрыть ноутбук, модель исчезнет. Нужно научиться сохранять её так, чтобы потом можно было:

- загрузить в другом скрипте;
- передать коллеге;
- развернуть на сервере (backend);
- использовать через месяц без переобучения.

В этом модуле мы разберём, как устроена обученная модель изнутри, как правильно сохранять её через `joblib`, и почему часто нужно сохранять не только модель, но и весь «окружающий мир» вокруг неё (скейлеры, энкодеры, метаданные).

## 1. Что именно мы сохраняем, когда сохраняем модель

### Обученная модель — это не только алгоритм

Когда вы пишете:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

переменная `model` — это объект в оперативной памяти. Но важно понимать: **до `fit()` и после `fit()` это два разных объекта**.

- **До `fit()`**: объект знает только гиперпараметры (`n_estimators=100`, `max_depth=None` и т.д.). Он «пустой».
- **После `fit()`**: объект наполнился **весами** — результатом обучения. Внутри него появились деревья решений, которые знают, как классифицировать новые данные.

### Где хранятся веса

В `sklearn` обученные параметры хранятся в атрибутах с **подчёркиванием в конце**. Это договорённость библиотеки:

In [ ]:
# После обучения
print(model.n_estimators)        # гиперпараметр (был известен до обучения)
print(model.feature_importances_) # обученный параметр (появился после fit)
print(model.classes_)             # классы, которые модель научилась различать

Если вы сохраняете модель — вы сохраняете **все эти атрибуты вместе**. Если бы вы сохранили только гиперпараметры (`n_estimators=100`), пришлось бы учить модель заново.

### Аналогия

Представьте, что модель — это пианист. Гиперпараметры — это характеристики пианино (количество клавиш, марка). А результат `fit()` — это **навыки пианиста**, выработанные годами практики. Сохранить только характеристики пианино бесполезно: вам нужен сам пианист со всеми его навыками.

## 2. Синтаксис `dump` и `load` в деталях

### `joblib.dump(obj, filename)`

Функция принимает два обязательных аргумента:

| Аргумент | Что это |
|----------|---------|
| `obj` | Любой Python-объект, который нужно сохранить |
| `filename` | Путь к файлу (строка). Расширение обычно `.joblib` |

In [ ]:
from joblib import dump

dump(model, 'random_forest_model.joblib')

После выполнения в рабочей папке появится файл `random_forest_model.joblib`.

### `joblib.load(filename)`

Функция принимает путь к файлу и возвращает исходный объект целиком:

In [ ]:
from joblib import load

model = load('random_forest_model.joblib')
prediction = model.predict(X_new)

### Важное правило: расширение не имеет значения

`joblib` не смотрит на расширение файла. Вы можете назвать файл `model.dat`, `model.pkl` или просто `model` — библиотека всё равно прочитает его правильно. Расширение `.joblib` — это договорённость для человека, чтобы сразу было понятно, чем открывать.

## 3. Сохранение одной модели sklearn

### Полный цикл: обучение -> сохранение -> загрузка -> предсказание

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from joblib import dump, load

# 1. Загружаем данные
iris = load_iris()
X, y = iris.data, iris.target

# 2. Делим на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Создаём и обучаем модель
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

# 4. Проверяем, что модель работает
print("Accuracy на тесте:", model.score(X_test, y_test))

# 5. СОХРАНЯЕМ модель
dump(model, 'iris_rf_model.joblib')
print("Модель сохранена!")

Теперь создадим **новый файл** (или перезапустим Python), как будто прошёл день:

In [ ]:
from joblib import load
import numpy as np

# 1. ЗАГРУЖАЕМ модель
model = load('iris_rf_model.joblib')

# 2. Проверяем, что она «живая»
print("Тип модели:", type(model))
print("Классы:", model.classes_)

# 3. Делаем предсказание на новых данных
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])  # одна ириска
prediction = model.predict(new_flower)
print("Предсказание:", prediction)

Модель загружена полностью: с гиперпараметрами, деревьями, всеми весами. Переобучать не нужно.

## 4. Почему сохранять одну модель — это ещё не всё

### Проблема: модель ожидает «подготовленные» данные

Редко модель работает с «сырыми» данными. Обычно перед подачей в модель данные проходят через цепочку преобразований:

In [ ]:
Сырые данные -> StandardScaler -> PCA -> Модель

Если вы сохранили только модель, но не сохранили `StandardScaler`, то при загрузке модели в другом месте вы не сможете правильно предобработать новые данные. Результат предсказания будет мусором.

### Решение: сохраняем всю цепочку

В `sklearn` есть класс `Pipeline`, который объединяет шаги предобработки и модель в один объект. Если сохранить `Pipeline` целиком — вы сохраните всю цепочку.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Создаём пайплайн
pipeline = Pipeline([
    ('scaler', StandardScaler()),           # шаг 1: масштабирование
    ('classifier', LogisticRegression())    # шаг 2: модель
])

# Обучаем ВЕСЬ пайплайн
pipeline.fit(X_train, y_train)

# Сохраняем ВЕСЬ пайплайн одним файлом
dump(pipeline, 'iris_pipeline.joblib')

При загрузке:

In [ ]:
pipeline = load('iris_pipeline.joblib')

# Можно подавать сырые данные — scaler сработает автоматически
prediction = pipeline.predict(new_raw_data)

> **Backend-правило:** всегда сохраняйте `Pipeline`, а не голую модель. Это защищает от ошибок несоответствия формата входных данных.

## 5. Сохранение нескольких объектов: словарь артефактов

Иногда нужно сохранить не только модель, но и дополнительные вещи: метрики качества, список фичей, версию библиотеки, дату обучения. Можно сохранить всё это в **один файл**, упаковав в словарь.

In [ ]:
from datetime import datetime

# Собираем всё в один словарь
artifacts = {
    'pipeline': pipeline,                           # обученный пайплайн
    'feature_names': iris.feature_names,            # имена признаков
    'target_names': iris.target_names,              # имена классов
    'metrics': {
        'accuracy': pipeline.score(X_test, y_test),
        'train_date': datetime.now().isoformat()
    },
    'model_version': '1.0.0'
}

# Сохраняем одним файлом
dump(artifacts, 'iris_experiment_artifacts.joblib')

При загрузке:

In [ ]:
artifacts = load('iris_experiment_artifacts.joblib')

model = artifacts['pipeline']
print("Accuracy при обучении:", artifacts['metrics']['accuracy'])
print("Признаки:", artifacts['feature_names'])
prediction = model.predict(new_data)

### Преимущества словаря артефактов

| Что сохраняем | Зачем |
|---------------|-------|
| Модель / Pipeline | Чтобы делать предсказания |
| Имена фичей | Чтобы проверить, что входные данные в правильном порядке |
| Метрики | Чтобы знать, насколько хороша модель, не переобучаясь |
| Дата и версия | Чтобы отслеживать эксперименты |

## 6. Формат `.joblib` и переносимость между компьютерами

### Что внутри файла

Файл `.joblib` — это **бинарный файл** (набор байтов). Внутри него:

- Структура объекта сериализуется через оптимизированный механизм (похожий на `pickle`, но с улучшениями для numpy).
- Большие numpy-массивы хранятся отдельными блоками для скорости.
- Может применяться сжатие (об этом в Модуле 2).

Вы не можете открыть `.joblib` в текстовом редакторе и прочитать — это не JSON и не CSV.

### Переносимость: главное правило

Файл `.joblib` **не является универсальным**. Он зависит от версий:

- **Python** (3.10 ≠ 3.12 может вызвать проблемы)
- **sklearn** (модель из `sklearn 1.3` может не загрузиться в `sklearn 1.5`)
- **joblib** (редко, но бывает)
- **numpy**, **scipy** (если модель их использует)

### Как делать правильно

1. **Фиксируйте версии** при сохранении:

In [ ]:
import sklearn
import joblib
import sys

artifacts = {
    'pipeline': pipeline,
    'versions': {
        'python': sys.version,
        'sklearn': sklearn.__version__,
        'joblib': joblib.__version__
    }
}
dump(artifacts, 'model_with_versions.joblib')

2. **Создавайте `requirements.txt`** рядом с моделью:

In [ ]:
pip freeze > requirements.txt

3. **Не скачивайте `.joblib` из неизвестных источников**. Как и `pickle`, `joblib.load()` может выполнить вредоносный код при загрузке.

### Перенос между Windows и Linux

Файлы `.joblib` обычно переносятся между операционными системами без проблем, **если версии библиотек совпадают**. Но пути к файлам внутри объекта (если вы их сохраняли) могут стать невалидными.

## 7. Практика: полный цикл «обучение -> деплой»

### Задание 1.1: «Сохрани и загрузи»

1. Загрузите датасет `Iris` (или любой другой из `sklearn.datasets`).
2. Разбейте данные на `train` и `test`.
3. Обучите `RandomForestClassifier`.
4. Сохраните модель в файл `my_first_model.joblib`.
5. Создайте **отдельный** Python-скрипт (или новую сессию), который:
   - загружает модель;
   - выводит тип загруженного объекта;
   - выводит значение `model.n_estimators` и `model.classes_`;
   - делает предсказание на одном примере из тестовой выборки.

### Задание 1.2: «Пайплайн целиком»

1. Создайте `Pipeline` из двух шагов: `StandardScaler` + `LogisticRegression`.
2. Обучите пайплайн на датасете `Iris`.
3. Сохраните пайплайн в файл `iris_full_pipeline.joblib`.
4. Загрузите пайплайн в новом скрипте.
5. Подайте на вход **сырые** данные (без масштабирования) и убедитесь, что предсказание работает.

### Задание 1.3: «Артефакты эксперимента»

1. Обучите любую модель.
2. Создайте словарь, содержащий:
   - саму модель;
   - список имён признаков (`iris.feature_names`);
   - словарь с метрикой `accuracy` на тесте;
   - строку с текущей датой.
3. Сохраните словарь в `experiment_v1.joblib`.
4. Загрузите и выведите на экран всё, кроме самой модели (имена фичей, метрики, дату).

## 8. Эталонное решение

### Решение 1.2 (Pipeline)

**Файл 1: `train.py`**

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from joblib import dump

# Данные
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# Пайплайн
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=200))
])
pipeline.fit(X_train, y_train)

# Сохраняем
dump(pipeline, 'iris_full_pipeline.joblib')
print("Pipeline сохранён. Accuracy:", pipeline.score(X_test, y_test))

**Файл 2: `predict.py`**

In [ ]:
from joblib import load
import numpy as np

# Загружаем
pipeline = load('iris_full_pipeline.joblib')

# Сырые данные (без масштабирования!)
sample = np.array([[5.0, 3.4, 1.5, 0.2]])

# Предсказание — scaler отработает внутри пайплайна
prediction = pipeline.predict(sample)
print("Предсказанный класс:", prediction)

## 9. Вопросы для самопроверки

1. **Что появляется у объекта модели после вызова `.fit()`?**  
   *(Ответ: обученные параметры — атрибуты с подчёркиванием на конце, например `coef_`, `feature_importances_`, `classes_`.)*

2. **Почему опасно сохранять только «голую» модель, без скейлера?**  
   *(Ответ: потому что модель обучена на масштабированных/преобразованных данных, а новые данные будут в другом масштабе — предсказания исказятся.)*

3. **Что будет, если загрузить `.joblib` в окружении с другой версией sklearn?**  
   *(Ответ: может возникнуть ошибка или модель будет работать некорректно. Версии библиотек должны совпадать.)*

4. **Зачем сохранять метаданные (имена фичей, метрики, дату) вместе с моделью?**  
   *(Ответ: для воспроизводимости эксперимента, отладки и понимания, какая это модель и насколько она хороша.)*

5. **Можно ли открыть `.joblib` в текстовом редакторе?**  
   *(Ответ: нет, это бинарный формат. Текстовый редактор покажет нечитаемые символы.)*

## Итоги модуля

- Обученная модель sklearn — это объект, наполненный **весами** после `fit()`. Сохранить нужно именно этот объект.
- `joblib.dump()` и `joblib.load()` — простой способ сохранить и восстановить любой Python-объект, включая модели.
- В ML лучше сохранять не голую модель, а **весь Pipeline** (предобработка + модель), чтобы не нарушить формат входных данных.
- Через словарь можно сохранить **несколько объектов** в один файл: модель, метрики, метаданные.
- Файлы `.joblib` зависят от версий библиотек. Для production всегда фиксируйте `requirements.txt` и сохраняйте версии вместе с моделью.

**В следующем модуле** мы научимся делать файлы меньше и загружать их быстрее — разберём сжатие и memory mapping.